In [1]:
import requests
import csv
import json
import time
import os
from pathlib import Path
from datetime import datetime, timedelta
from xml.etree import ElementTree as ET
from dotenv import load_dotenv

# Load API key from .env
load_dotenv(dotenv_path=r"D:\Shikha\bias_project\.env")
API_KEY = os.getenv("GOVINFO_API_KEY")

if not API_KEY:
    raise ValueError("GOVINFO_API_KEY not found in .env")

print(f"API key loaded (length: {len(API_KEY)} chars)")

# === Configuration ===
BASE_URL = "https://api.govinfo.gov"
TARGET_YEAR = 2024
TARGET_MONTH = 1

# Output paths
OUTPUT_DIR = Path(r"D:\Shikha\bias_project\corpus\congressional_record\2024-01")
SPEECHES_CSV = OUTPUT_DIR / "speeches.csv"
PROGRESS_LOG = OUTPUT_DIR / "progress.log"
ERRORS_LOG = OUTPUT_DIR / "errors.log"

# Rate limiting
REQUEST_DELAY_SEC = 1.0
RETRY_DELAY_SEC = 15
MAX_RETRIES = 3

# Verify output dir exists
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Output directory: {OUTPUT_DIR}")
print(f"Output exists:    {OUTPUT_DIR.exists()}")
print(f"Target:           January {TARGET_YEAR}")

API key loaded (length: 40 chars)
Output directory: D:\Shikha\bias_project\corpus\congressional_record\2024-01
Output exists:    True
Target:           January 2024


In [2]:
def call_api(endpoint, params=None, max_retries=MAX_RETRIES):
    """
    Make a GET request to the GovInfo API with rate limiting and retry logic.
    
    Returns (status_code, json_data_or_text) tuple.
    """
    if params is None:
        params = {}
    
    # Always include API key
    params["api_key"] = API_KEY
    
    url = endpoint if endpoint.startswith("http") else f"{BASE_URL}{endpoint}"
    
    for attempt in range(max_retries):
        try:
            # Polite delay before request
            time.sleep(REQUEST_DELAY_SEC)
            
            response = requests.get(url, params=params, timeout=30)
            
            # Success
            if response.status_code == 200:
                # Try JSON first, fall back to text
                try:
                    return 200, response.json()
                except ValueError:
                    return 200, response.text
            
            # Rate limited - back off and retry
            elif response.status_code == 429:
                wait = RETRY_DELAY_SEC * (attempt + 1)
                print(f"  Rate limited (429). Waiting {wait}s before retry {attempt + 1}/{max_retries}...")
                time.sleep(wait)
                continue
            
            # Other client/server errors - retry with backoff
            elif response.status_code >= 500:
                wait = RETRY_DELAY_SEC * (attempt + 1)
                print(f"  Server error ({response.status_code}). Waiting {wait}s before retry {attempt + 1}/{max_retries}...")
                time.sleep(wait)
                continue
            
            # 4xx errors other than 429 - don't retry
            else:
                return response.status_code, response.text[:500]
        
        except requests.exceptions.Timeout:
            print(f"  Timeout on attempt {attempt + 1}/{max_retries}. Retrying...")
            time.sleep(RETRY_DELAY_SEC)
            continue
        
        except requests.exceptions.RequestException as e:
            print(f"  Request error: {type(e).__name__}: {str(e)[:100]}")
            time.sleep(RETRY_DELAY_SEC)
            continue
    
    return None, f"Failed after {max_retries} retries"


# Test the helper with a simple call
print("Testing API helper...")
status, data = call_api("/collections")

if status == 200:
    coll_count = len(data.get('collections', []))
    print(f"✓ Helper works. Status: {status}, collections: {coll_count}")
else:
    print(f"✗ Helper test failed. Status: {status}")
    print(f"  Response: {data[:200] if isinstance(data, str) else data}")

Testing API helper...
✓ Helper works. Status: 200, collections: 41


In [3]:
# Find all CREC packages issued in January 2024
# We'll walk day-by-day since the /collections/CREC/{start}/{end} endpoint 
# turned out to be unreliable in our earlier testing

from datetime import date, timedelta

start_date = date(2024, 1, 1)
end_date = date(2024, 1, 31)

packages_found = []
current = start_date

print(f"Checking each day from {start_date} to {end_date}...\n")

while current <= end_date:
    package_id = f"CREC-{current.strftime('%Y-%m-%d')}"
    
    # Check if this package exists by fetching its summary
    status, data = call_api(f"/packages/{package_id}/summary")
    
    if status == 200:
        packages_found.append({
            'package_id': package_id,
            'date': current.isoformat(),
            'title': data.get('title', ''),
            'pages': data.get('pages', 0),
            'congress': data.get('congress', ''),
            'session': data.get('session', ''),
        })
        print(f"  {package_id}  ✓  {data.get('pages', 0)} pages")
    else:
        # Most likely no session that day (404)
        print(f"  {package_id}  -  (no session, status {status})")
    
    current += timedelta(days=1)

print(f"\n--- Summary ---")
print(f"Days checked:        {(end_date - start_date).days + 1}")
print(f"Session days found:  {len(packages_found)}")
print(f"Total pages:         {sum(p['pages'] for p in packages_found)}")

Checking each day from 2024-01-01 to 2024-01-31...

  CREC-2024-01-01  -  (no session, status 404)
  CREC-2024-01-02  ✓  10 pages
  CREC-2024-01-03  -  (no session, status 404)
  CREC-2024-01-04  -  (no session, status 404)
  CREC-2024-01-05  ✓  8 pages
  CREC-2024-01-06  -  (no session, status 404)
  CREC-2024-01-07  -  (no session, status 404)
  CREC-2024-01-08  ✓  24 pages
  CREC-2024-01-09  ✓  51 pages
  CREC-2024-01-10  ✓  80 pages
  CREC-2024-01-11  ✓  94 pages
  CREC-2024-01-12  ✓  31 pages
  CREC-2024-01-13  -  (no session, status 404)
  CREC-2024-01-14  -  (no session, status 404)
  CREC-2024-01-15  -  (no session, status 404)
  CREC-2024-01-16  ✓  46 pages
  CREC-2024-01-17  ✓  105 pages
  CREC-2024-01-18  ✓  91 pages
  CREC-2024-01-19  -  (no session, status 404)
  CREC-2024-01-20  -  (no session, status 404)
  CREC-2024-01-21  -  (no session, status 404)
  CREC-2024-01-22  ✓  23 pages
  CREC-2024-01-23  ✓  26 pages
  CREC-2024-01-24  ✓  32 pages
  CREC-2024-01-25  ✓  42 pag

TypeError: unsupported operand type(s) for +: 'int' and 'str'

In [4]:
# Re-compute the summary with type-safe page counting
total_pages = 0
type_issues = 0

for p in packages_found:
    pages = p['pages']
    if isinstance(pages, int):
        total_pages += pages
    elif isinstance(pages, str):
        try:
            total_pages += int(pages)
        except (ValueError, TypeError):
            type_issues += 1
    else:
        type_issues += 1

print(f"--- Summary ---")
print(f"Days checked:        31")
print(f"Session days found:  {len(packages_found)}")
print(f"Total pages:         {total_pages}")
print(f"Pages field issues:  {type_issues}")

# Show what we actually have
print(f"\nSession days in January 2024:")
for p in packages_found:
    print(f"  {p['package_id']}  {p['pages']} pages  ({p['title'][:60]})")

--- Summary ---
Days checked:        31
Session days found:  18
Total pages:         869
Pages field issues:  0

Session days in January 2024:
  CREC-2024-01-02  10 pages  (Congressional Record Volume 169, Issue 214, (January 2, 2024)
  CREC-2024-01-05  8 pages  (Congressional Record Volume 170, Issue 2, (January 5, 2024))
  CREC-2024-01-08  24 pages  (Congressional Record Volume 170, Issue 3, (January 8, 2024))
  CREC-2024-01-09  51 pages  (Congressional Record Volume 170, Issue 4, (January 9, 2024))
  CREC-2024-01-10  80 pages  (Congressional Record Volume 170, Issue 5, (January 10, 2024))
  CREC-2024-01-11  94 pages  (Congressional Record Volume 170, Issue 6, (January 11, 2024))
  CREC-2024-01-12  31 pages  (Congressional Record Volume 170, Issue 7, (January 12, 2024))
  CREC-2024-01-16  46 pages  (Congressional Record Volume 170, Issue 8, (January 16, 2024))
  CREC-2024-01-17  105 pages  (Congressional Record Volume 170, Issue 9, (January 17, 2024))
  CREC-2024-01-18  91 pages  (Co

In [5]:
# For each session day, fetch the list of granules (individual speeches/items)
# We collect them all into a flat list for processing in Cell 5

all_granules = []
days_processed = 0
days_total = len(packages_found)

print(f"Fetching granules for {days_total} session days...\n")

for pkg in packages_found:
    package_id = pkg['package_id']
    days_processed += 1
    
    # Get the granules list for this package
    # Paginate if needed (use offsetMark for pagination)
    granules_for_day = []
    offset_mark = "*"
    
    while True:
        params = {
            "pageSize": 100,
            "offsetMark": offset_mark,
        }
        status, data = call_api(f"/packages/{package_id}/granules", params=params)
        
        if status != 200:
            print(f"  {package_id}: ERROR fetching granules (status {status})")
            break
        
        granules = data.get('granules', [])
        granules_for_day.extend(granules)
        
        # Check if there are more pages
        next_mark = data.get('nextPage')
        if not next_mark or len(granules) == 0:
            break
        offset_mark = next_mark
    
    # Add date metadata to each granule for later use
    for g in granules_for_day:
        g['_package_date'] = pkg['date']
        g['_package_id'] = package_id
    
    all_granules.extend(granules_for_day)
    print(f"  [{days_processed}/{days_total}] {package_id}: {len(granules_for_day)} granules")

print(f"\n--- Summary ---")
print(f"Total granules across {days_total} days: {len(all_granules)}")

# Show class distribution (HOUSE / SENATE / EXTENSIONS / DAILYDIGEST etc.)
from collections import Counter
class_counts = Counter(g.get('granuleClass', 'UNKNOWN') for g in all_granules)
print(f"\nGranule class distribution:")
for cls, count in class_counts.most_common():
    print(f"  {cls:20s}  {count}")

Fetching granules for 18 session days...

  [1/18] CREC-2024-01-02: 29 granules
  [2/18] CREC-2024-01-05: 32 granules
  [3/18] CREC-2024-01-08: 43 granules
  Server error (500). Waiting 15s before retry 1/3...
  Server error (500). Waiting 30s before retry 2/3...
  Server error (500). Waiting 45s before retry 3/3...
  CREC-2024-01-09: ERROR fetching granules (status None)
  [4/18] CREC-2024-01-09: 100 granules
  Server error (500). Waiting 15s before retry 1/3...
  Server error (500). Waiting 30s before retry 2/3...
  Server error (500). Waiting 45s before retry 3/3...
  CREC-2024-01-10: ERROR fetching granules (status None)
  [5/18] CREC-2024-01-10: 100 granules
  Server error (500). Waiting 15s before retry 1/3...
  Server error (500). Waiting 30s before retry 2/3...
  Server error (500). Waiting 45s before retry 3/3...
  CREC-2024-01-11: ERROR fetching granules (status None)
  [6/18] CREC-2024-01-11: 100 granules
  [7/18] CREC-2024-01-12: 79 granules
  Server error (500). Waiting 15

In [6]:
from collections import Counter

# How many granules did each day return?
day_counts = Counter()
for g in all_granules:
    day_counts[g['_package_id']] += 1

print("Granules per day (sorted by count):")
print(f"{'Date':<22} {'Count':<8} {'Likely truncated?'}")
print("-" * 60)

for pkg_id, count in sorted(day_counts.items(), key=lambda x: -x[1]):
    truncated = "⚠️ YES" if count == 100 else "no"
    print(f"  {pkg_id:<22} {count:<8} {truncated}")

print(f"\nTotal granules collected: {sum(day_counts.values())}")
print(f"Days where pagination may have stopped at 100: {sum(1 for c in day_counts.values() if c == 100)}")

Granules per day (sorted by count):
Date                   Count    Likely truncated?
------------------------------------------------------------
  CREC-2024-01-09        100      ⚠️ YES
  CREC-2024-01-10        100      ⚠️ YES
  CREC-2024-01-11        100      ⚠️ YES
  CREC-2024-01-16        100      ⚠️ YES
  CREC-2024-01-17        100      ⚠️ YES
  CREC-2024-01-18        100      ⚠️ YES
  CREC-2024-01-25        100      ⚠️ YES
  CREC-2024-01-29        100      ⚠️ YES
  CREC-2024-01-30        100      ⚠️ YES
  CREC-2024-01-31        100      ⚠️ YES
  CREC-2024-01-22        92       no
  CREC-2024-01-12        79       no
  CREC-2024-01-23        63       no
  CREC-2024-01-24        60       no
  CREC-2024-01-08        43       no
  CREC-2024-01-05        32       no
  CREC-2024-01-02        29       no
  CREC-2024-01-26        11       no

Total granules collected: 1409
Days where pagination may have stopped at 100: 10


In [7]:
# Identify days that hit the 100-granule ceiling (likely truncated)
truncated_days = [pkg_id for pkg_id, count in day_counts.items() if count == 100]

print(f"Retrying pagination for {len(truncated_days)} truncated days...")
print(f"Days to retry: {truncated_days}\n")

# Remove existing granules from these days so we can re-fetch cleanly
all_granules = [g for g in all_granules if g['_package_id'] not in truncated_days]

# Re-fetch with longer pagination patience
for package_id in truncated_days:
    granules_for_day = []
    offset_mark = "*"
    page_num = 1
    
    while True:
        params = {
            "pageSize": 100,
            "offsetMark": offset_mark,
        }
        status, data = call_api(f"/packages/{package_id}/granules", params=params)
        
        if status != 200:
            print(f"  {package_id} page {page_num}: ERROR (status {status})")
            break
        
        granules = data.get('granules', [])
        granules_for_day.extend(granules)
        
        next_mark = data.get('nextPage')
        if not next_mark or len(granules) == 0:
            break
        offset_mark = next_mark
        page_num += 1
    
    # Find the original package metadata
    pkg = next((p for p in packages_found if p['package_id'] == package_id), None)
    if pkg:
        for g in granules_for_day:
            g['_package_date'] = pkg['date']
            g['_package_id'] = package_id
    
    all_granules.extend(granules_for_day)
    print(f"  {package_id}: {len(granules_for_day)} granules across {page_num} page(s)")

# Re-summarize
from collections import Counter
class_counts = Counter(g.get('granuleClass', 'UNKNOWN') for g in all_granules)
print(f"\n--- Updated Summary ---")
print(f"Total granules: {len(all_granules)}")
print(f"\nGranule class distribution:")
for cls, count in class_counts.most_common():
    print(f"  {cls:20s}  {count}")

Retrying pagination for 10 truncated days...
Days to retry: ['CREC-2024-01-09', 'CREC-2024-01-10', 'CREC-2024-01-11', 'CREC-2024-01-16', 'CREC-2024-01-17', 'CREC-2024-01-18', 'CREC-2024-01-25', 'CREC-2024-01-29', 'CREC-2024-01-30', 'CREC-2024-01-31']

  Server error (500). Waiting 15s before retry 1/3...
  Server error (500). Waiting 30s before retry 2/3...
  Server error (500). Waiting 45s before retry 3/3...
  CREC-2024-01-09 page 2: ERROR (status None)
  CREC-2024-01-09: 100 granules across 2 page(s)
  Server error (500). Waiting 15s before retry 1/3...
  Server error (500). Waiting 30s before retry 2/3...
  Server error (500). Waiting 45s before retry 3/3...
  CREC-2024-01-10 page 2: ERROR (status None)
  CREC-2024-01-10: 100 granules across 2 page(s)
  Server error (500). Waiting 15s before retry 1/3...
  Server error (500). Waiting 30s before retry 2/3...
  Server error (500). Waiting 45s before retry 3/3...
  CREC-2024-01-11 page 2: ERROR (status None)
  CREC-2024-01-11: 100 gra

In [8]:
# Inspect what the API returns as nextPage so we know what's malformed
# Use a day we know has more than 100 granules

test_package = "CREC-2024-01-09"
params = {"pageSize": 100, "offsetMark": "*"}
status, data = call_api(f"/packages/{test_package}/granules", params=params)

print(f"Status: {status}")
print(f"\nKeys in response: {list(data.keys()) if isinstance(data, dict) else 'not a dict'}")
print(f"\nGranules returned: {len(data.get('granules', []))}")
print(f"Count (total): {data.get('count')}")
print(f"nextPage value: {data.get('nextPage')}")
print(f"previousPage value: {data.get('previousPage')}")

Status: 200

Keys in response: ['count', 'offset', 'pageSize', 'nextPage', 'previousPage', 'granules']

Granules returned: 100
Count (total): 128
nextPage value: https://api.govinfo.gov/packages/CREC-2024-01-09/granules?offsetMark=AoM2MTcwLzAwNC8wMS8xLzAwMDA0OS8wMyIwMT8MQ1JFQy0yMDI0LTAxLTA5LUNSRUMtMjAyNC0wMS0wOS1wdDEtUGdTNDktMw%3D%3D&pageSize=100
previousPage value: None


In [9]:
# Identify days that hit the 100-granule ceiling and re-fetch with proper URL pagination
from collections import Counter

day_counts = Counter()
for g in all_granules:
    day_counts[g['_package_id']] += 1

truncated_days = [pkg_id for pkg_id, count in day_counts.items() if count == 100]

print(f"Re-fetching {len(truncated_days)} truncated days with corrected pagination...\n")

# Remove existing partial data from these days
all_granules = [g for g in all_granules if g['_package_id'] not in truncated_days]

for package_id in truncated_days:
    granules_for_day = []
    page_num = 1
    
    # First page
    params = {"pageSize": 100, "offsetMark": "*"}
    status, data = call_api(f"/packages/{package_id}/granules", params=params)
    
    if status != 200:
        print(f"  {package_id} page 1: ERROR (status {status})")
        continue
    
    granules_for_day.extend(data.get('granules', []))
    total_count = data.get('count', 0)
    next_url = data.get('nextPage')
    
    # Subsequent pages — call next_url directly without adding params
    # (the URL already has offsetMark and pageSize)
    while next_url:
        page_num += 1
        status, data = call_api(next_url, params={})  # no params, URL has them
        
        if status != 200:
            print(f"  {package_id} page {page_num}: ERROR (status {status})")
            break
        
        new_granules = data.get('granules', [])
        granules_for_day.extend(new_granules)
        
        next_url = data.get('nextPage')
        if not next_url or len(new_granules) == 0:
            break
    
    # Attach metadata
    pkg = next((p for p in packages_found if p['package_id'] == package_id), None)
    if pkg:
        for g in granules_for_day:
            g['_package_date'] = pkg['date']
            g['_package_id'] = package_id
    
    all_granules.extend(granules_for_day)
    print(f"  {package_id}: {len(granules_for_day)}/{total_count} granules across {page_num} page(s)")

# Updated summary
class_counts = Counter(g.get('granuleClass', 'UNKNOWN') for g in all_granules)
print(f"\n--- Updated Summary ---")
print(f"Total granules: {len(all_granules)}")
print(f"\nGranule class distribution:")
for cls, count in class_counts.most_common():
    print(f"  {cls:20s}  {count}")

Re-fetching 10 truncated days with corrected pagination...

  CREC-2024-01-09: 128/128 granules across 2 page(s)
  CREC-2024-01-10: 183/183 granules across 2 page(s)
  CREC-2024-01-11: 216/216 granules across 3 page(s)
  CREC-2024-01-16: 101/101 granules across 2 page(s)
  CREC-2024-01-17: 172/172 granules across 2 page(s)
  CREC-2024-01-18: 210/210 granules across 3 page(s)
  CREC-2024-01-25: 145/145 granules across 2 page(s)
  CREC-2024-01-29: 119/119 granules across 2 page(s)
  CREC-2024-01-30: 165/165 granules across 2 page(s)
  CREC-2024-01-31: 197/197 granules across 2 page(s)

--- Updated Summary ---
Total granules: 2045

Granule class distribution:
  HOUSE                 886
  SENATE                642
  EXTENSIONS            400
  DAILYDIGEST           117


In [10]:
# Filter to floor speech granules (HOUSE and SENATE only)
floor_granules = [
    g for g in all_granules 
    if g.get('granuleClass') in ('HOUSE', 'SENATE')
]

print(f"Floor speech granules: {len(floor_granules)}")
print(f"Will fetch speaker metadata and text for each.\n")
print(f"This will take a while — ~1 second per granule.")
print(f"Estimated time: {len(floor_granules) * 1.5 / 60:.0f}–{len(floor_granules) * 2 / 60:.0f} minutes\n")

Floor speech granules: 1528
Will fetch speaker metadata and text for each.

This will take a while — ~1 second per granule.
Estimated time: 38–51 minutes



In [11]:
# Extract speaker metadata and speech text for each floor granule
# Writes to CSV incrementally so we can resume if interrupted

import csv
from bs4 import BeautifulSoup

CSV_FIELDS = [
    'date', 'package_id', 'granule_id', 'granule_class', 'title',
    'speaker_name', 'state', 'party', 'chamber', 'bioguide_id',
    'speaker_chamber', 'text', 'text_length'
]

# Load already-processed granule IDs if CSV exists (for resume)
processed_ids = set()
if SPEECHES_CSV.exists():
    with open(SPEECHES_CSV, 'r', encoding='utf-8', newline='') as f:
        reader = csv.DictReader(f)
        for row in reader:
            processed_ids.add(row['granule_id'])
    print(f"Found existing CSV with {len(processed_ids)} already-processed granules")

# Open CSV in append mode (creates if missing)
file_exists = SPEECHES_CSV.exists()
csv_file = open(SPEECHES_CSV, 'a', encoding='utf-8', newline='')
writer = csv.DictWriter(csv_file, fieldnames=CSV_FIELDS)
if not file_exists:
    writer.writeheader()

# Counters
stats = {
    'attempted': 0,
    'skipped_already_done': 0,
    'no_speakers': 0,
    'multi_speaker': 0,
    'no_text': 0,
    'extracted': 0,
    'errors': 0,
}

# Open error log
errors_file = open(ERRORS_LOG, 'a', encoding='utf-8')

start_time = time.time()
total = len(floor_granules)

print(f"\nStarting extraction of {total} granules...")
print(f"Output: {SPEECHES_CSV}\n")

for i, gran in enumerate(floor_granules, 1):
    granule_id = gran['granuleId']
    
    # Skip if already processed
    if granule_id in processed_ids:
        stats['skipped_already_done'] += 1
        continue
    
    stats['attempted'] += 1
    package_id = gran['_package_id']
    
    # Fetch granule summary (has speaker metadata)
    status, summary = call_api(f"/packages/{package_id}/granules/{granule_id}/summary")
    
    if status != 200:
        stats['errors'] += 1
        errors_file.write(f"{granule_id}\tsummary_failed\tstatus={status}\n")
        continue
    
    members = summary.get('members', [])
    
    # We want single-speaker granules for clean state attribution
    if len(members) == 0:
        stats['no_speakers'] += 1
        continue
    if len(members) > 1:
        stats['multi_speaker'] += 1
        continue
    
    member = members[0]
    
    # Fetch text content via txtLink
    download_links = summary.get('download', {})
    txt_link = download_links.get('txtLink')
    
    if not txt_link:
        stats['no_text'] += 1
        continue
    
    status, text_html = call_api(txt_link)
    
    if status != 200 or not isinstance(text_html, str):
        stats['errors'] += 1
        errors_file.write(f"{granule_id}\ttext_fetch_failed\tstatus={status}\n")
        continue
    
    # Strip HTML to get clean text
    soup = BeautifulSoup(text_html, 'html.parser')
    text = soup.get_text(separator=' ', strip=True)
    
    # Write to CSV immediately (so we don't lose progress)
    row = {
        'date': gran.get('_package_date', ''),
        'package_id': package_id,
        'granule_id': granule_id,
        'granule_class': gran.get('granuleClass', ''),
        'title': summary.get('title', ''),
        'speaker_name': member.get('memberName', ''),
        'state': member.get('state', ''),
        'party': member.get('party', ''),
        'chamber': member.get('chamber', ''),
        'bioguide_id': member.get('bioGuideId', ''),
        'speaker_chamber': member.get('chamber', ''),
        'text': text,
        'text_length': len(text),
    }
    writer.writerow(row)
    csv_file.flush()  # write to disk immediately
    
    stats['extracted'] += 1
    
    # Progress update every 50 granules
    if i % 50 == 0:
        elapsed_min = (time.time() - start_time) / 60
        rate = i / elapsed_min if elapsed_min > 0 else 0
        eta_min = (total - i) / rate if rate > 0 else 0
        print(f"  [{i}/{total}] elapsed {elapsed_min:.1f}m, rate {rate:.0f}/min, "
              f"ETA {eta_min:.0f}m | extracted {stats['extracted']}, "
              f"skipped {stats['no_speakers']+stats['multi_speaker']}, errors {stats['errors']}")

# Close files
csv_file.close()
errors_file.close()

elapsed_min = (time.time() - start_time) / 60

print(f"\n--- Extraction Complete ---")
print(f"Total time:           {elapsed_min:.1f} minutes")
print(f"Granules attempted:   {stats['attempted']}")
print(f"Already done (skipped): {stats['skipped_already_done']}")
print(f"Successfully extracted: {stats['extracted']}")
print(f"No speakers:          {stats['no_speakers']}")
print(f"Multi-speaker (skipped): {stats['multi_speaker']}")
print(f"No text link:         {stats['no_text']}")
print(f"Errors:               {stats['errors']}")
print(f"\nOutput written to:    {SPEECHES_CSV}")
print(f"Errors logged to:     {ERRORS_LOG}")# Extract speaker metadata and speech text for each floor granule
# Writes to CSV incrementally so we can resume if interrupted

import csv
from bs4 import BeautifulSoup

CSV_FIELDS = [
    'date', 'package_id', 'granule_id', 'granule_class', 'title',
    'speaker_name', 'state', 'party', 'chamber', 'bioguide_id',
    'speaker_chamber', 'text', 'text_length'
]

# Load already-processed granule IDs if CSV exists (for resume)
processed_ids = set()
if SPEECHES_CSV.exists():
    with open(SPEECHES_CSV, 'r', encoding='utf-8', newline='') as f:
        reader = csv.DictReader(f)
        for row in reader:
            processed_ids.add(row['granule_id'])
    print(f"Found existing CSV with {len(processed_ids)} already-processed granules")

# Open CSV in append mode (creates if missing)
file_exists = SPEECHES_CSV.exists()
csv_file = open(SPEECHES_CSV, 'a', encoding='utf-8', newline='')
writer = csv.DictWriter(csv_file, fieldnames=CSV_FIELDS)
if not file_exists:
    writer.writeheader()

# Counters
stats = {
    'attempted': 0,
    'skipped_already_done': 0,
    'no_speakers': 0,
    'multi_speaker': 0,
    'no_text': 0,
    'extracted': 0,
    'errors': 0,
}

# Open error log
errors_file = open(ERRORS_LOG, 'a', encoding='utf-8')

start_time = time.time()
total = len(floor_granules)

print(f"\nStarting extraction of {total} granules...")
print(f"Output: {SPEECHES_CSV}\n")

for i, gran in enumerate(floor_granules, 1):
    granule_id = gran['granuleId']
    
    # Skip if already processed
    if granule_id in processed_ids:
        stats['skipped_already_done'] += 1
        continue
    
    stats['attempted'] += 1
    package_id = gran['_package_id']
    
    # Fetch granule summary (has speaker metadata)
    status, summary = call_api(f"/packages/{package_id}/granules/{granule_id}/summary")
    
    if status != 200:
        stats['errors'] += 1
        errors_file.write(f"{granule_id}\tsummary_failed\tstatus={status}\n")
        continue
    
    members = summary.get('members', [])
    
    # We want single-speaker granules for clean state attribution
    if len(members) == 0:
        stats['no_speakers'] += 1
        continue
    if len(members) > 1:
        stats['multi_speaker'] += 1
        continue
    
    member = members[0]
    
    # Fetch text content via txtLink
    download_links = summary.get('download', {})
    txt_link = download_links.get('txtLink')
    
    if not txt_link:
        stats['no_text'] += 1
        continue
    
    status, text_html = call_api(txt_link)
    
    if status != 200 or not isinstance(text_html, str):
        stats['errors'] += 1
        errors_file.write(f"{granule_id}\ttext_fetch_failed\tstatus={status}\n")
        continue
    
    # Strip HTML to get clean text
    soup = BeautifulSoup(text_html, 'html.parser')
    text = soup.get_text(separator=' ', strip=True)
    
    # Write to CSV immediately (so we don't lose progress)
    row = {
        'date': gran.get('_package_date', ''),
        'package_id': package_id,
        'granule_id': granule_id,
        'granule_class': gran.get('granuleClass', ''),
        'title': summary.get('title', ''),
        'speaker_name': member.get('memberName', ''),
        'state': member.get('state', ''),
        'party': member.get('party', ''),
        'chamber': member.get('chamber', ''),
        'bioguide_id': member.get('bioGuideId', ''),
        'speaker_chamber': member.get('chamber', ''),
        'text': text,
        'text_length': len(text),
    }
    writer.writerow(row)
    csv_file.flush()  # write to disk immediately
    
    stats['extracted'] += 1
    
    # Progress update every 50 granules
    if i % 50 == 0:
        elapsed_min = (time.time() - start_time) / 60
        rate = i / elapsed_min if elapsed_min > 0 else 0
        eta_min = (total - i) / rate if rate > 0 else 0
        print(f"  [{i}/{total}] elapsed {elapsed_min:.1f}m, rate {rate:.0f}/min, "
              f"ETA {eta_min:.0f}m | extracted {stats['extracted']}, "
              f"skipped {stats['no_speakers']+stats['multi_speaker']}, errors {stats['errors']}")

# Close files
csv_file.close()
errors_file.close()

elapsed_min = (time.time() - start_time) / 60

print(f"\n--- Extraction Complete ---")
print(f"Total time:           {elapsed_min:.1f} minutes")
print(f"Granules attempted:   {stats['attempted']}")
print(f"Already done (skipped): {stats['skipped_already_done']}")
print(f"Successfully extracted: {stats['extracted']}")
print(f"No speakers:          {stats['no_speakers']}")
print(f"Multi-speaker (skipped): {stats['multi_speaker']}")
print(f"No text link:         {stats['no_text']}")
print(f"Errors:               {stats['errors']}")
print(f"\nOutput written to:    {SPEECHES_CSV}")
print(f"Errors logged to:     {ERRORS_LOG}")


Starting extraction of 1528 granules...
Output: D:\Shikha\bias_project\corpus\congressional_record\2024-01\speeches.csv

  [100/1528] elapsed 5.0m, rate 20/min, ETA 72m | extracted 33, skipped 67, errors 0
  [250/1528] elapsed 12.3m, rate 20/min, ETA 63m | extracted 79, skipped 171, errors 0
  [400/1528] elapsed 19.9m, rate 20/min, ETA 56m | extracted 130, skipped 270, errors 0
  [550/1528] elapsed 27.5m, rate 20/min, ETA 49m | extracted 185, skipped 365, errors 0
  [800/1528] elapsed 40.6m, rate 20/min, ETA 37m | extracted 286, skipped 514, errors 0
  [900/1528] elapsed 46.2m, rate 19/min, ETA 32m | extracted 322, skipped 578, errors 0
  [1200/1528] elapsed 63.4m, rate 19/min, ETA 17m | extracted 429, skipped 771, errors 0
  [1250/1528] elapsed 65.9m, rate 19/min, ETA 15m | extracted 442, skipped 808, errors 0
  [1350/1528] elapsed 71.6m, rate 19/min, ETA 9m | extracted 475, skipped 875, errors 0
  [1400/1528] elapsed 74.8m, rate 19/min, ETA 7m | extracted 506, skipped 894, errors 0


Error: field larger than field limit (131072)

In [13]:
import csv
import pandas as pd

# Use a large but C-long-safe value (10 MB per field, way more than any speech)
csv.field_size_limit(10_000_000)

df = pd.read_csv(SPEECHES_CSV, engine='python')

print(f"Total speeches: {len(df)}")
print(f"Total text characters: {df['text_length'].sum():,}")
print(f"Estimated total words: {df['text_length'].sum() / 5:,.0f}")
print(f"Average speech length: {df['text_length'].mean():.0f} chars")
print(f"Longest speech: {df['text_length'].max():,} chars")
print(f"Shortest speech: {df['text_length'].min()} chars")

print(f"\n--- State coverage ---")
state_counts = df['state'].value_counts()
print(f"Unique states: {len(state_counts)}")

print(f"\nTop 10 states by speech count:")
print(state_counts.head(10))

print(f"\nBottom 10 states by speech count:")
print(state_counts.tail(10))

print(f"\n--- Chamber breakdown ---")
print(df['chamber'].value_counts())

print(f"\n--- Party breakdown ---")
print(df['party'].value_counts())

Total speeches: 547
Total text characters: 1,988,632
Estimated total words: 397,726
Average speech length: 3636 chars
Longest speech: 242,825 chars
Shortest speech: 346 chars

--- State coverage ---
Unique states: 51

Top 10 states by speech count:
state
NY    112
CA     45
PA     26
IL     23
NC     23
KY     18
OR     17
MO     16
OH     16
FL     15
Name: count, dtype: int64

Bottom 10 states by speech count:
state
HI    2
GU    1
AK    1
IN    1
OK    1
VI    1
MS    1
NH    1
DE    1
ME    1
Name: count, dtype: int64

--- Chamber breakdown ---
chamber
H    293
S    254
Name: count, dtype: int64

--- Party breakdown ---
party
D    311
R    232
I      4
Name: count, dtype: int64


In [14]:
print("Full state distribution (sorted by count):")
print(df['state'].value_counts().to_string())

Full state distribution (sorted by count):
state
NY    112
CA     45
PA     26
IL     23
NC     23
KY     18
OR     17
MO     16
OH     16
FL     15
TX     14
NJ     14
IA     13
TN     12
MD     11
MN     11
VA     10
GA     10
KS      9
SC      9
MA      9
MI      8
VT      8
WI      8
CT      7
AL      7
UT      7
AR      7
AZ      6
RI      6
NM      5
ID      4
CO      4
NV      4
WA      4
LA      4
SD      4
ND      3
NE      3
WY      2
MT      2
HI      2
GU      1
AK      1
IN      1
OK      1
VI      1
MS      1
NH      1
DE      1
ME      1
